# AIRMAN Aeronautics — Data Science Assessment
## Skynet + TOGA Intelligence Analysis
**Candidate:** Data Science Intern  
**Date:** 2026-06-10  
**Products:** Skynet (FTO Operations SaaS) + TOGA (Pilot Learning App)

---
This notebook contains the complete analysis for the AIRMAN Data Science Assessment.  
It is structured into 8 tasks as specified in the assessment document.

## Setup — Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# ── Load all datasets ──────────────────────────────────────────────────────
sorties     = pd.read_csv('data/sorties.csv', parse_dates=['scheduled_start','scheduled_end','actual_start','actual_end'])
aircraft    = pd.read_csv('data/aircraft.csv')
cadets      = pd.read_csv('data/cadets.csv', parse_dates=['enrollment_date'])
instructors = pd.read_csv('data/instructors.csv')
toga        = pd.read_csv('data/toga_study.csv', parse_dates=['last_active_date'])
payments    = pd.read_csv('data/payments.csv', parse_dates=['last_payment_date'])

print("✅ All datasets loaded successfully")
print(f"  Sorties: {len(sorties)} rows | Aircraft: {len(aircraft)} | Cadets: {len(cadets)}")
print(f"  Instructors: {len(instructors)} | TOGA records: {len(toga)} | Payments: {len(payments)}")

---
## Task 1 — Data Cleaning and Validation

In [ ]:
# ── Validation Check 1: Completed sorties must have actual times ──────────
mask_complete_no_time = (sorties['status'] == 'completed') & (sorties['actual_start'].isna() | sorties['actual_end'].isna())
print(f"Completed sorties missing actual times: {mask_complete_no_time.sum()}")

# ── Validation Check 2: Cancelled sorties should NOT have actual times ────
mask_cancel_has_time = (sorties['status'] == 'cancelled') & (~sorties['actual_start'].isna())
print(f"Cancelled sorties with actual times (unexpected): {mask_cancel_has_time.sum()}")

# ── Validation Check 3: Delay minutes vs actual calculation ──────────────
comp = sorties[sorties['status']=='completed'].copy()
comp['calc_delay'] = ((comp['actual_start'] - comp['scheduled_start']).dt.total_seconds() / 60).round(0)
mismatch = comp[abs(comp['calc_delay'] - comp['delay_minutes']) > 1]
print(f"Delay minutes mismatch (>1 min difference): {len(mismatch)} sorties → {mismatch['sortie_id'].tolist()}")

# ── Validation Check 4: Payment math consistency ──────────────────────────
payments['calc_outstanding'] = payments['invoiced_amount'] - payments['paid_amount']
pay_err = payments[abs(payments['calc_outstanding'] - payments['outstanding_amount']) > 0]
print(f"Payment calculation errors: {len(pay_err)}")

# ── Validation Check 5: Study progress > 100% ─────────────────────────────
toga['study_progress_pct'] = toga['chapters_completed'] / toga['total_chapters'] * 100
over100 = toga[toga['study_progress_pct'] > 100]
print(f"Study progress > 100%: {len(over100)} records")

# ── Validation Check 6: Maintenance downtime vs available hours ───────────
ac_bad = aircraft[aircraft['maintenance_downtime_hours'] > aircraft['total_available_hours']]
print(f"Aircraft where downtime > available hours: {len(ac_bad)}")

# ── Validation Check 7: High defect count ────────────────────────────────
high_defect = aircraft[aircraft['defect_count'] >= 8]
print(f"\nHigh defect aircraft (>=8): {high_defect[['aircraft_id','registration','defect_count']].to_string(index=False)}")

# ── Validation Check 8: Duplicate IDs ────────────────────────────────────
print(f"\nDuplicate sortie_ids: {sorties.duplicated('sortie_id').sum()}")
print(f"Duplicate cadet_ids: {cadets.duplicated('cadet_id').sum()}")

# ── Missing values summary ────────────────────────────────────────────────
print("\n── Missing Values Summary ──")
for name, df in [('sorties',sorties),('aircraft',aircraft),('cadets',cadets),('toga',toga),('payments',payments)]:
    mv = df.isnull().sum()
    mv = mv[mv>0]
    print(f"  {name}: {mv.to_dict() if len(mv) else 'No missing values'}")

---
## Task 2 — Skynet Operations Analytics

In [ ]:
# ── Aircraft Utilization ──────────────────────────────────────────────────
sorties['actual_duration_hrs'] = (sorties['actual_end'] - sorties['actual_start']).dt.total_seconds() / 3600
ac_flown = sorties[sorties['status']=='completed'].groupby('aircraft_id')['actual_duration_hrs'].sum().reset_index()
ac_flown.columns = ['aircraft_id','actual_flown_hours']

aircraft_analysis = aircraft.merge(ac_flown, on='aircraft_id', how='left').fillna({'actual_flown_hours':0})
aircraft_analysis['utilization_pct'] = (aircraft_analysis['actual_flown_hours'] / aircraft_analysis['total_available_hours'] * 100).round(2)

print("── Aircraft Utilization ──")
print(aircraft_analysis[['aircraft_id','registration','type','total_available_hours','actual_flown_hours','utilization_pct','defect_count']].to_string(index=False))

# ── Instructor Utilization ────────────────────────────────────────────────
instructors['flight_to_duty_ratio'] = (instructors['total_flight_hours'] / instructors['total_duty_hours'] * 100).round(2)
print("\n── Instructor Utilization ──")
print(instructors[['name','base_id','total_duty_hours','total_flight_hours','flight_to_duty_ratio']].to_string(index=False))

# ── Dispatch Reliability ──────────────────────────────────────────────────
total   = len(sorties)
comp    = (sorties['status']=='completed').sum()
canc    = (sorties['status']=='cancelled').sum()
delayed = ((sorties['status']=='completed') & (sorties['delay_minutes']>0)).sum()
avg_dly = sorties[sorties['status']=='completed']['delay_minutes'].mean()

print(f"\n── Dispatch Reliability ──")
print(f"Total sorties: {total} | Completed: {comp} ({comp/total*100:.1f}%) | Cancelled: {canc} ({canc/total*100:.1f}%)")
print(f"Delayed (completed but late): {delayed} | Avg delay: {avg_dly:.1f} min")
print("\nCancellation Reasons:")
print(sorties[sorties['status']=='cancelled']['cancel_reason'].value_counts().to_string())

---
## Task 3 — Training Progress Analytics

In [ ]:
today = pd.Timestamp('2026-06-10')

cadets['progress_pct'] = (cadets['total_flown_hours'] / cadets['total_required_hours'] * 100).round(2)
cadets['remaining_hours'] = cadets['total_required_hours'] - cadets['total_flown_hours']
cadets['days_enrolled'] = (today - cadets['enrollment_date']).dt.days
cadets['avg_flying_rate_hrs_per_week'] = (cadets['total_flown_hours'] / (cadets['days_enrolled']/7)).round(2)
cadets['est_weeks_to_complete'] = (cadets['remaining_hours'] / cadets['avg_flying_rate_hrs_per_week']).round(1)
cadets['completion_risk'] = cadets['est_weeks_to_complete'].apply(
    lambda x: 'High' if x > 52 else ('Medium' if x > 26 else 'Low'))

print("── Cadet Training Progress ──")
cols = ['name','course','progress_pct','remaining_hours','avg_flying_rate_hrs_per_week','est_weeks_to_complete','completion_risk']
print(cadets[cols].to_string(index=False))

print("""
── Key Insight ──
C002 (Meera Iyer) is 37% through CPL, has ₹1,10,000 outstanding payments,
low TOGA study performance (avg quiz score 51.5), and repeated disruptions.
Training completion risk is HIGH.

C003 (Rahul Nair) has only 12 hrs flown in 110 days. Aircraft A003 defects
caused 2 direct cancellations. Combined with high payment risk, this is CRITICAL.""")

---
## Task 4 — TOGA Study Intelligence

In [ ]:
toga['study_progress_pct'] = (toga['chapters_completed'] / toga['total_chapters'] * 100).round(2)
toga['days_inactive'] = (today - toga['last_active_date']).dt.days
toga['practice_score'] = (toga['practice_tests_attempted'] / toga['practice_tests_attempted'].max() * 100).round(2)
toga['subject_readiness'] = (
    0.4 * toga['study_progress_pct'] +
    0.4 * toga['avg_quiz_score'] +
    0.2 * toga['practice_score']
).round(2)

print("── Subject Readiness Scores ──")
id_name = {'C001':'Arjun Menon','C002':'Meera Iyer','C003':'Rahul Nair'}
toga['cadet_name'] = toga['cadet_id'].map(id_name)
print(toga[['cadet_name','subject','study_progress_pct','avg_quiz_score','subject_readiness','days_inactive']].to_string(index=False))

toga_cadet = toga.groupby('cadet_id').agg(
    avg_study_progress=('study_progress_pct','mean'),
    avg_quiz_score=('avg_quiz_score','mean'),
    total_practice_tests=('practice_tests_attempted','sum'),
    avg_readiness=('subject_readiness','mean'),
    max_inactive_days=('days_inactive','max')
).round(2).reset_index()

print("\n── Cadet-Level TOGA Summary ──")
toga_cadet['name'] = toga_cadet['cadet_id'].map(id_name)
print(toga_cadet[['name','avg_study_progress','avg_quiz_score','avg_readiness','max_inactive_days']].to_string(index=False))

print("""
── Key Insight ──
C003 has low Navigation progress (16.7%), quiz score 42, inactive 43 days.
Recommend targeted Navigation revision and immediate instructor check-in.""")

---
## Task 5 — Finance and Operational Risk

In [ ]:
payments['payment_pct'] = (payments['paid_amount'] / payments['invoiced_amount'] * 100).round(2)
payments['days_since_payment'] = (today - payments['last_payment_date']).dt.days

def payment_risk(row):
    score = 0
    if row['outstanding_amount'] > 80000: score += 2
    elif row['outstanding_amount'] > 30000: score += 1
    if row['days_since_payment'] > 40: score += 2
    elif row['days_since_payment'] > 20: score += 1
    if row['payment_pct'] < 75: score += 1
    return 'High' if score >= 4 else ('Medium' if score >= 2 else 'Low')

payments['payment_risk'] = payments.apply(payment_risk, axis=1)
pay_display = payments.merge(cadets[['cadet_id','name']], on='cadet_id')

print("── Finance Risk Analysis ──")
print(pay_display[['name','invoiced_amount','paid_amount','outstanding_amount','payment_pct','days_since_payment','payment_risk']].to_string(index=False))

print(f"\nTotal Outstanding: ₹{payments['outstanding_amount'].sum():,}")
print(f"Collection Rate: {payments['paid_amount'].sum()/payments['invoiced_amount'].sum()*100:.1f}%")

---
## Task 6 — Explainable Cadet Risk Score

In [ ]:
# ── Merge all data for risk scoring ─────────────────────────────────────
cancel_counts = sorties[sorties['status']=='cancelled'].groupby('cadet_id').size().reset_index(name='cancel_count')
avg_delays = sorties[sorties['status']=='completed'].groupby('cadet_id')['delay_minutes'].mean().reset_index(name='avg_delay')

risk_df = cadets[['cadet_id','name','progress_pct']].copy()
risk_df = risk_df.merge(toga_cadet[['cadet_id','avg_study_progress','avg_quiz_score','avg_readiness','max_inactive_days']], on='cadet_id', how='left')
risk_df = risk_df.merge(payments[['cadet_id','outstanding_amount','payment_risk','payment_pct']], on='cadet_id', how='left')
risk_df = risk_df.merge(cancel_counts, on='cadet_id', how='left').fillna({'cancel_count':0})
risk_df = risk_df.merge(avg_delays, on='cadet_id', how='left').fillna({'avg_delay':0})

# ── Risk Score Formula ──────────────────────────────────────────────────
def compute_risk_score(row):
    flight_risk      = (1 - min(row['progress_pct']/100, 1)) * 25
    study_pct        = row['avg_study_progress'] if not pd.isna(row['avg_study_progress']) else 0
    study_risk       = (1 - min(study_pct/100, 1)) * 20
    quiz             = row['avg_quiz_score'] if not pd.isna(row['avg_quiz_score']) else 0
    quiz_risk        = (1 - min(quiz/100, 1)) * 15
    inactive         = row['max_inactive_days'] if not pd.isna(row['max_inactive_days']) else 60
    inactivity_risk  = min(inactive/60, 1) * 15
    pay_pct          = row['payment_pct'] if not pd.isna(row['payment_pct']) else 0
    payment_risk_sc  = (1 - min(pay_pct/100, 1)) * 15
    cancel_risk      = min(row['cancel_count']/3, 1) * 5
    delay_risk       = min(row['avg_delay']/40, 1) * 5
    return round(flight_risk + study_risk + quiz_risk + inactivity_risk + payment_risk_sc + cancel_risk + delay_risk, 1)

def main_reasons(row):
    r = []
    if row['progress_pct'] < 40: r.append("Low flight progress")
    if not pd.isna(row['avg_study_progress']) and row['avg_study_progress'] < 50: r.append("Low study progress")
    if not pd.isna(row['avg_quiz_score']) and row['avg_quiz_score'] < 55: r.append("Low quiz score")
    if not pd.isna(row['max_inactive_days']) and row['max_inactive_days'] > 30: r.append("Study inactivity")
    if not pd.isna(row['payment_pct']) and row['payment_pct'] < 80: r.append("High payment risk")
    if row['cancel_count'] >= 2: r.append("Frequent cancellations")
    if row['avg_delay'] > 20: r.append("High avg delay")
    return ", ".join(r) if r else "Performing well"

risk_df['risk_score'] = risk_df.apply(compute_risk_score, axis=1)
risk_df['risk_level'] = risk_df['risk_score'].apply(lambda s: 'Low' if s<40 else ('Medium' if s<70 else 'High'))
risk_df['main_reasons'] = risk_df.apply(main_reasons, axis=1)

print("── Cadet Risk Scores ──")
print(risk_df[['name','risk_score','risk_level','main_reasons']].to_string(index=False))

# Save risk scores
risk_df[['cadet_id','name','risk_score','risk_level','main_reasons']].to_csv('data/risk_scores.csv', index=False)
print("\n✅ risk_scores.csv saved")

---
## Task 7 — Visualizations

In [ ]:
# Chart 1: Aircraft Utilization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Aircraft Utilization Analysis', fontsize=16, fontweight='bold')

ax1 = axes[0]
bars = ax1.bar(aircraft_analysis['registration'], aircraft_analysis['utilization_pct'],
               color=['#1f77b4','#ff7f0e','#d62728'], edgecolor='black', linewidth=0.5)
ax1.set_title('Aircraft Utilization Rate (%)'); ax1.set_ylabel('Utilization (%)')
ax1.axhline(50, color='red', linestyle='--', label='50% Benchmark', linewidth=1.5)
for bar, val in zip(bars, aircraft_analysis['utilization_pct']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'{val:.1f}%', ha='center', fontweight='bold')
ax1.legend()

ax2 = axes[1]
x = np.arange(len(aircraft_analysis)); w = 0.35
ax2.bar(x-w/2, aircraft_analysis['total_available_hours'], w, label='Available Hours', color='#1f77b4', edgecolor='black')
ax2.bar(x+w/2, aircraft_analysis['maintenance_downtime_hours'], w, label='Maintenance Downtime', color='#d62728', edgecolor='black')
ax2.set_xticks(x); ax2.set_xticklabels(aircraft_analysis['registration'])
ax2.set_title('Available Hours vs Maintenance Downtime'); ax2.set_ylabel('Hours'); ax2.legend()
plt.tight_layout()
plt.savefig('charts/aircraft_utilization.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 1: Aircraft Utilization saved")

In [ ]:
# Chart 2: Cancellation Reasons
cancel_reasons = sorties[sorties['status']=='cancelled']['cancel_reason'].value_counts()
fig, ax = plt.subplots(figsize=(9,6))
cancel_reasons.plot(kind='bar', ax=ax, color=['#d62728','#ff7f0e','#9467bd'], edgecolor='black')
ax.set_title('Sortie Cancellation Reasons', fontsize=15, fontweight='bold')
ax.set_xlabel('Reason'); ax.set_ylabel('Count'); ax.tick_params(axis='x', rotation=20)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x()+p.get_width()/2, p.get_height()+0.05), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('charts/cancellation_reasons.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 2: Cancellation Reasons saved")

In [ ]:
# Chart 3: Cadet Flight Progress
fig, ax = plt.subplots(figsize=(10,6))
c_sorted = cadets.sort_values('progress_pct', ascending=False)
bar_colors = ['#2ca02c' if p>=60 else '#ff7f0e' if p>=30 else '#d62728' for p in c_sorted['progress_pct']]
bars = ax.barh(c_sorted['name'], c_sorted['progress_pct'], color=bar_colors, edgecolor='black')
ax.set_title('Cadet Flight Progress (%)', fontsize=15, fontweight='bold'); ax.set_xlabel('Progress (%)')
for bar, val, flown, req in zip(bars, c_sorted['progress_pct'], c_sorted['total_flown_hours'], c_sorted['total_required_hours']):
    ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2, f'{val:.1f}% ({flown}/{req} hrs)', va='center', fontsize=10)
patches = [mpatches.Patch(color='#2ca02c',label='On Track (≥60%)'),
           mpatches.Patch(color='#ff7f0e',label='At Risk (30-60%)'),
           mpatches.Patch(color='#d62728',label='Critical (<30%)')]
ax.legend(handles=patches)
plt.tight_layout()
plt.savefig('charts/cadet_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 3: Cadet Progress saved")

In [ ]:
# Chart 4: TOGA Study Readiness
toga_pivot = toga.pivot_table(index='cadet_id', columns='subject', values='subject_readiness', aggfunc='mean').fillna(0)
toga_pivot.index = toga_pivot.index.map({'C001':'Arjun Menon','C002':'Meera Iyer','C003':'Rahul Nair'})
fig, ax = plt.subplots(figsize=(10,6))
toga_pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black')
ax.set_title('TOGA Study Readiness Score by Subject', fontsize=15, fontweight='bold')
ax.set_xlabel('Cadet'); ax.set_ylabel('Readiness Score (0-100)')
ax.axhline(60, color='red', linestyle='--', label='Min Readiness (60)', linewidth=1.5)
ax.legend(loc='upper right'); ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('charts/study_readiness.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 4: Study Readiness saved")

In [ ]:
# Chart 5: Payment Status
pay_names = payments.merge(cadets[['cadet_id','name']], on='cadet_id')
x = np.arange(len(pay_names)); w = 0.35
fig, ax = plt.subplots(figsize=(10,6))
ax.bar(x-w/2, pay_names['paid_amount']/1000, w, label='Paid (₹K)', color='#2ca02c', edgecolor='black')
ax.bar(x+w/2, pay_names['outstanding_amount']/1000, w, label='Outstanding (₹K)', color='#d62728', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(pay_names['name'])
ax.set_title('Cadet Payment Status', fontsize=15, fontweight='bold'); ax.set_ylabel('Amount (₹ Thousands)')
for i,(paid,out) in enumerate(zip(pay_names['paid_amount'],pay_names['outstanding_amount'])):
    ax.text(i-w/2,paid/1000+2,f'₹{paid//1000}K',ha='center',fontsize=9)
    ax.text(i+w/2,out/1000+2,f'₹{out//1000}K',ha='center',fontsize=9)
ax.legend(); plt.tight_layout()
plt.savefig('charts/payment_risk.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 5: Payment Risk saved")

In [ ]:
# Chart 6: Cadet Risk Scores
risk_plot = risk_df.sort_values('risk_score', ascending=False)
risk_colors = ['#d62728' if r=='High' else '#ff7f0e' if r=='Medium' else '#2ca02c' for r in risk_plot['risk_level']]
fig, ax = plt.subplots(figsize=(10,6))
bars = ax.barh(risk_plot['name'], risk_plot['risk_score'], color=risk_colors, edgecolor='black')
ax.set_title('Cadet Risk Score (0–100)', fontsize=15, fontweight='bold'); ax.set_xlabel('Risk Score')
ax.axvline(40, color='#ff7f0e', linestyle='--', label='Medium Threshold (40)', linewidth=1.5)
ax.axvline(70, color='#d62728', linestyle='--', label='High Threshold (70)', linewidth=1.5)
for bar, score, level in zip(bars, risk_plot['risk_score'], risk_plot['risk_level']):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2, f'{score} ({level})', va='center', fontsize=11, fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('charts/cadet_risk_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 6: Cadet Risk Scores saved")

In [ ]:
# Chart 7: Flight Progress vs Study Progress Scatter
merged_s = cadets[['cadet_id','name','progress_pct']].merge(toga_cadet[['cadet_id','avg_study_progress','avg_quiz_score']],on='cadet_id')
merged_s = merged_s.merge(risk_df[['cadet_id','risk_level']],on='cadet_id')
scatter_colors = {'High':'#d62728','Medium':'#ff7f0e','Low':'#2ca02c'}
fig, ax = plt.subplots(figsize=(9,7))
for _, row in merged_s.iterrows():
    ax.scatter(row['progress_pct'], row['avg_study_progress'],
               s=row['avg_quiz_score']*8, color=scatter_colors[row['risk_level']], alpha=0.8, edgecolors='black')
    ax.annotate(row['name'], (row['progress_pct'], row['avg_study_progress']),
                textcoords='offset points', xytext=(10,5), fontsize=11, fontweight='bold')
ax.set_title('Flight Progress vs Study Progress\n(Bubble size = Quiz Score)', fontsize=14, fontweight='bold')
ax.set_xlabel('Flight Progress (%)'); ax.set_ylabel('Avg Study Progress (%)')
ax.axhline(50, color='gray', linestyle='--', alpha=0.5); ax.axvline(50, color='gray', linestyle=':', alpha=0.5)
patches = [mpatches.Patch(color=v,label=k+' Risk') for k,v in scatter_colors.items()]
ax.legend(handles=patches, loc='lower right'); plt.tight_layout()
plt.savefig('charts/flight_vs_study_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 7: Flight vs Study Progress saved")

---
## Summary — All Tasks Complete

All reports, charts, and data files have been generated. See `reports/` and `charts/` directories.